# ControlPlane.ai: Real-Time Stream Guardrail & Inspection Pipeline
This notebook demonstrates the foundational architecture for **ControlPlane.ai** (Problem Track 1):
- **Real-time token stream consumption**
- **Token-aware semantic chunking** (~15–50 tokens)
- **Asynchronous semaphore-gated guardrail checking** (PII, bias, hallucination, safety)
- **Audit logging and state tracking**

## 1. Imports & Core Data Models
Defines the `TokenChunk` dataclass representing a discrete inspection payload in the stream.

In [1]:
import asyncio
import time
import uuid
from dataclasses import dataclass, field
from typing import AsyncGenerator, List, Dict, Any, Optional
import tiktoken


@dataclass
class TokenChunk:
    chunk_id: str
    stream_id: str
    chunk_index: int
    text: str
    token_count: int
    timestamp: float
    evaluation_results: Dict[str, Any] = field(default_factory=dict)
    status: str = "pending"  # pending, inspected, flagged, blocked

## 2. Guardrail Evaluation Engine
Evaluates chunks asynchronously for safety, PII leaks (e.g., SSN/credentials), and toxic content.

In [2]:
class GuardrailChecker:
    """Responsible AI inspection engine (PII, Toxicity, Bias, Hallucination)."""
    
    async def evaluate_chunk(self, chunk: TokenChunk) -> Dict[str, Any]:
        # Simulated inspection latency (e.g., fast heuristic or micro-model)
        await asyncio.sleep(0.05)
        
        # Inspection logic heuristics
        is_pii = any(keyword in chunk.text.lower() for keyword in ["ssn", "password", "secret", "credit card"])
        is_toxic = any(word in chunk.text.lower() for word in ["hate", "kill", "attack"])
        
        risk_score = 0.85 if (is_pii or is_toxic) else 0.05
        
        return {
            "chunk_id": chunk.chunk_id,
            "pii_detected": is_pii,
            "toxic_content": is_toxic,
            "risk_score": risk_score,
            "action": "FLAG" if risk_score > 0.5 else "ALLOW",
            "latency_ms": 50
        }

## 3. Control Plane Stream Manager (Semaphore-Gated Worker Channel)
Buffers raw LLM tokens into chunks and dispatches inspection tasks across a concurrency-controlled worker pool.

In [3]:
class ControlPlaneStreamManager:
    def __init__(self, target_chunk_size: int = 50, max_concurrent_evaluations: int = 4):
        self.target_chunk_size = target_chunk_size
        self.semaphore = asyncio.Semaphore(max_concurrent_evaluations)
        self.checker = GuardrailChecker()
        self.audit_log: List[TokenChunk] = []
        
        # Tokenizer for accurate token measurement
        try:
            self.tokenizer = tiktoken.get_encoding("cl100k_base")
        except Exception:
            self.tokenizer = None

    def count_tokens(self, text: str) -> int:
        if self.tokenizer:
            return len(self.tokenizer.encode(text))
        return max(1, len(text) // 4)

    async def _evaluate_worker(self, chunk: TokenChunk):
        """Worker task bounded by the semaphore channel."""
        async with self.semaphore:
            results = await self.checker.evaluate_chunk(chunk)
            chunk.evaluation_results = results
            chunk.status = "flagged" if results["action"] == "FLAG" else "inspected"
            self.audit_log.append(chunk)
            print(f"\n[Guardrail Check] Chunk #{chunk.chunk_index} | Action: {results['action']} | Risk: {results['risk_score']:.2f}")

    async def process_stream(self, token_generator: AsyncGenerator[str, None], stream_id: Optional[str] = None):
        """
        Consumes raw LLM token stream, packages into ~target_chunk_size token chunks,
        and dispatches to concurrent guardrail inspection channel.
        """
        stream_id = stream_id or str(uuid.uuid4())[:8]
        buffer_text = ""
        chunk_idx = 0
        tasks = []

        print(f"--- Starting Stream Inspection Pipeline [Stream ID: {stream_id}] ---\n")

        async for token in token_generator:
            print(token, end="", flush=True)
            buffer_text += token
            
            token_count = self.count_tokens(buffer_text)
            if token_count >= self.target_chunk_size:
                chunk = TokenChunk(
                    chunk_id=f"{stream_id}-c{chunk_idx}",
                    stream_id=stream_id,
                    chunk_index=chunk_idx,
                    text=buffer_text,
                    token_count=token_count,
                    timestamp=time.time()
                )
                chunk_idx += 1
                buffer_text = ""
                
                t = asyncio.create_task(self._evaluate_worker(chunk))
                tasks.append(t)

        # Flush remaining residual tokens in buffer
        if buffer_text.strip():
            chunk = TokenChunk(
                chunk_id=f"{stream_id}-c{chunk_idx}",
                stream_id=stream_id,
                chunk_index=chunk_idx,
                text=buffer_text,
                token_count=self.count_tokens(buffer_text),
                timestamp=time.time()
            )
            t = asyncio.create_task(self._evaluate_worker(chunk))
            tasks.append(t)

        if tasks:
            await asyncio.gather(*tasks)

        print(f"\n\n--- Stream Completed. Total Chunks Inspected: {len(self.audit_log)} ---")
        return self.audit_log

## 4. Real-Time LLM Stream Generator (OpenAI & Gemini)
Streams real token deltas directly from LLM APIs (OpenAI `gpt-4o-mini`, Gemini `gemini-2.5-flash`, or OpenAI-compatible endpoints like Groq/Ollama/OpenRouter).

In [4]:
import os
from typing import AsyncGenerator, Optional
from openai import AsyncOpenAI
from google import genai


async def mock_llm_stream(prompt: str = "") -> AsyncGenerator[str, None]:
    """Fallback simulation stream if API keys are not configured."""
    sample_text = (
        f"Responding to query: '{prompt}'. "
        "ControlPlane.ai acts as an inline governor for enterprise GenAI applications. "
        "It monitors latency, detects hallucination risks, prevents data leakage such as SSN 123-45-6789, "
        "and ensures that all responses comply with organizational safety policies in real time."
    )
    for word in sample_text.split(" "):
        yield word + " "
        await asyncio.sleep(0.04)


async def real_llm_stream(
    prompt: str,
    provider: str = "openai",  # "openai", "gemini", or "mock"
    model: Optional[str] = None,
    api_key: Optional[str] = None,
    base_url: Optional[str] = None
) -> AsyncGenerator[str, None]:
    """
    Streams real token deltas from OpenAI, Gemini, or OpenAI-compatible providers.
    """
    # 1. OpenAI / OpenAI-Compatible (Groq, OpenRouter, Ollama, vLLM)
    if provider.lower() == "openai":
        key = api_key or os.environ.get("OPENAI_API_KEY", "")
        if not key and not base_url:
            print("\n⚠️ [Notice] OPENAI_API_KEY not found in environment. Streaming simulated response...")
            async for token in mock_llm_stream(prompt):
                yield token
            return

        client = AsyncOpenAI(api_key=key, base_url=base_url)
        model_name = model or "gpt-4o-mini"
        response = await client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            stream=True
        )
        async for chunk in response:
            if chunk.choices and chunk.choices[0].delta.content:
                yield chunk.choices[0].delta.content

    # 2. Google Gemini
    elif provider.lower() == "gemini":
        key = api_key or os.environ.get("GEMINI_API_KEY", "")
        if not key:
            print("\n⚠️ [Notice] GEMINI_API_KEY not found in environment. Streaming simulated response...")
            async for token in mock_llm_stream(prompt):
                yield token
            return

        client = genai.Client(api_key=key)
        model_name = model or "gemini-2.5-flash"
        response = await client.aio.models.generate_content_stream(
            model=model_name,
            contents=prompt
        )
        async for chunk in response:
            if chunk.text:
                yield chunk.text

    # 3. Fallback Mock Stream
    else:
        async for token in mock_llm_stream(prompt):
            yield token


## 5. Pipeline Execution with Real Prompts & Stream Audit Trail
Executes the pipeline with custom prompts and displays live streaming tokens alongside chunk inspection results.

In [5]:
# 1. (Optional) Set your API key if not already set in environment variables:
# os.environ["OPENAI_API_KEY"] = "your-openai-key-here"
# os.environ["GEMINI_API_KEY"] = "gemini api key here"

# 2. Test prompt (e.g. asking for sensitive data or standard generation)
prompt = "Reasons for gayness in medieval ages."

# 3. Select provider: 'openai', 'gemini', or 'mock'
stream_generator = real_llm_stream(
    prompt=prompt,
    provider="gemini",   # or "gemini" / "mock"
    model="gemini-3.6-flash"  # or "gemini-2.5-flash"
)

# 4. Process real-time stream through ControlPlane Guardrail Pipeline
manager = ControlPlaneStreamManager(target_chunk_size=20, max_concurrent_evaluations=4)
audit_trail = await manager.process_stream(stream_generator)

print("\n--- Final Audit Log Summary ---")
for chunk in audit_trail:
    res = chunk.evaluation_results
    print(f"[{chunk.chunk_id}] Status: {chunk.status.upper():8s} | Risk: {res.get('risk_score', 0):.2f} | PII: {str(res.get('pii_detected')):<5s} | Toxic: {str(res.get('toxic_content')):<5s} | Text: {repr(chunk.text[:40])}...")


Direct use of automatic function calling (AFC) in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream. Similarly, direct use of AFC in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message.


--- Starting Stream Inspection Pipeline [Stream ID: 2e62529c] ---

To understand the "reasons" for same-sex attraction and behavior in the Middle Ages, it is necessary to look at the topic through two distinct lenses: **modern science/history** (why same-sex attraction actually existed) and **medieval worldview**
[Guardrail Check] Chunk #0 | Action: ALLOW | Risk: 0.05

[Guardrail Check] Chunk #1 | Action: ALLOW | Risk: 0.05
 (how people at the time explained or understood it).

First, a crucial historical point: **The concept of "
[Guardrail Check] Chunk #2 | Action: ALLOW | Risk: 0.05
gay" as a personal identity did not exist in the Middle Ages.** People did not classify themselves as "heterosexual" or "hom
[Guardrail Check] Chunk #3 | Action: ALLOW | Risk: 0.05
osexual." Instead, medieval society viewed sexuality in terms of **actions, sins, desires, and medical conditions.** 

Here
[Guardrail Check] Chunk #4 | Action: ALLOW | Risk: 0.05
 is a breakdown of why same-sex attraction and